In [1]:
!git clone https://github.com/structuredllm/itergen.git

Cloning into 'itergen'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (563/563), done.
remote: Compressing objects: 100% (241/241), done.
remote: Total 563 (delta 201), reused 559 (delta 198), pack-reused 0 (from 0)
Receiving objects: 100% (563/563), 4.83 MiB | 5.54 MiB/s, done.
Resolving deltas: 100% (201/201), done.


In [ ]:
%pip install setuptools==80.9.0 wheel
%pip install --no-build-isolation -r itergen/requirements.txt

In [3]:
import sys, os
from pathlib import Path

repo = Path("itergen").resolve()

# Надо поменять
cache_dir = Path("/mnt/d/Test-task-RP-cache")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["SYNCODE_CACHE"] = str(cache_dir) + os.sep
os.environ["HF_CACHE"] = str(cache_dir) + os.sep

for path in [
    repo,
    repo / "itergen" / "syncode",
    repo / "itergen" / "syncode" / "syncode",
]:
    sys.path.insert(0, str(path))
sys.modules.pop("itergen", None)

In [4]:
import torch
import transformers
from itergen import Dataset
from itergen.syncode.syncode.infer import Syncode
from types import MethodType
from syncode.grammar_decoder import SyncodeLogitsProcessor
from syncode.parsers.grammars import Grammar

sql_grammar = Grammar("sql")
sql_dataset = Dataset("spider", language="sql")
device = "cuda" if torch.cuda.is_available() else "cpu"

''' обычная генерация '''
baseline = Syncode(
    model="Qwen/Qwen3.5-0.8B",
    mode="original",
    grammar="sql",
    parse_output_only=True,
    quantize=False,
    device=device,
    do_sample=False,
    max_new_tokens=200,
)
if device.startswith("cuda"):
    baseline.model.model.to(dtype=torch.float16)
baseline.model.model.eval()
print("Модель загружена. Примеров:", len(sql_dataset.problems))

''' Syncode генерация  '''
sql_decoder = SyncodeLogitsProcessor(
    grammar=sql_grammar,
    use_cache=True,
    tokenizer=baseline.model.tokenizer,
    parse_output_only=True,
    num_samples=1,
    mode="grammar_strict",
    dev_mode=True,
)
print("Грамматический процессор SynCode готов")

/home/nikita/proj/Test-task-RP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 2066.90it/s]


Модель загружена. Примеров: 1034
Loading DFA mask store from /mnt/d/Test-task-RP-cache/mask_stores/TokenizersBackend/grammar_strict_6057483450_248044.pkl.
DFA mask store loaded from cache.
Грамматический процессор SynCode готов


In [5]:
MAX_NEW_TOKENS = 200
STOP_WORDS = ("\n\n",)

@torch.inference_mode()
def _generate(messages, max_new_tokens=MAX_NEW_TOKENS,
              stop_words=STOP_WORDS, decoder=None):
    tokenizer = baseline.model.tokenizer
    model = baseline.model.model

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(next(model.parameters()).device)
    prompt_length = inputs["input_ids"].shape[1]

    processors = transformers.LogitsProcessorList()
    if decoder is not None:
        decoder.reset("")
        decoder.start_from = prompt_length
        processors.append(decoder)

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        repetition_penalty=1.0,
        logits_processor=processors,
        stop_strings=list(stop_words) if stop_words else None,
        tokenizer=tokenizer,
        return_dict_in_generate=False,
    )

    generated_ids = output[:, prompt_length:]
    raw_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return raw_output, generated_ids.shape[1]


def generate_original(messages, max_new_tokens=MAX_NEW_TOKENS,
                      stop_words=STOP_WORDS):
    return _generate(
        messages, max_new_tokens=max_new_tokens, stop_words=stop_words,
        decoder=None,
    )


def generate_syncode(messages, max_new_tokens=MAX_NEW_TOKENS,
                     stop_words=STOP_WORDS):
    return _generate(
        messages, max_new_tokens=max_new_tokens, stop_words=stop_words,
        decoder=sql_decoder,
    )

In [ ]:
import random

SEED = 67
r = random.Random(SEED)
main_db = {}

for source_id, problem in enumerate(sql_dataset.problems):
    db_id = problem["db_id"]
    example = dict(problem, source_id=source_id)

    if db_id not in main_db:
        main_db[db_id] = []
        
    main_db[db_id].append(example)

extra_examples_db = []

for db_id in main_db:
    examples = main_db[db_id]

    if len(examples) >= 11:
        extra_examples_db.append(db_id)

extra_examples_db.sort()
assert len(extra_examples_db) >= 5, "Недостаточно подходящих баз"

''' Рандомно выбираем 5 баз, и только те, где есть хотя бы 11 примеров (10 на базовые примеры и 1 на отладку) '''
selected_databases = r.sample(extra_examples_db, 5)

eval_examples = []
debug_examples = []
for db_id in selected_databases:
    examples = main_db[db_id]
    selected = r.sample(examples, 11)
    for i in range(10):
        eval_examples.append(selected[i])
    
    debug_examples.append(selected[10])

r.shuffle(eval_examples)
r.shuffle(debug_examples)
assert len(eval_examples) == 50
assert len(debug_examples) == 5

# проверка на то, что у нас отладка не попала в основу
eval_ids = []

for example in eval_examples:
    eval_ids.append(example["source_id"])
for example in debug_examples:
    assert example["source_id"] not in eval_ids

print("Основная выборка:", len(eval_examples))
print("Отладочная выборка:", len(debug_examples))
print("Распределение по базам:")

counts = {}
for db_id in selected_databases:
    counts[db_id] = 0
for example in eval_examples:
    db_id = example["db_id"]
    counts[db_id] += 1
for db_id in counts:
    print(db_id, "-", counts[db_id])

    

Основная выборка: 50
Отладочная выборка: 5
Распределение по базам:
concert_singer — 10
course_teach — 10
singer — 10
student_transcripts_tracking — 10
employee_hire_evaluation — 10


In [ ]:
import sqlite3
import time
import urllib.request
import zipfile
from pathlib import Path

DB_ROOT = cache_dir / "spider_databases"
SPIDER_ARCHIVE = cache_dir / "spider_data.zip"
SPIDER_URL = "https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip"
selected_db_ids = set()
for example in eval_examples:
    selected_db_ids.add(example["db_id"])
for example in debug_examples:
    selected_db_ids.add(example["db_id"])

missing_db_ids = []
for db_id in sorted(selected_db_ids):
    db_file = DB_ROOT / db_id / f"{db_id}.sqlite"
    if not db_file.is_file():
        missing_db_ids.append(db_id)

if missing_db_ids:
    if not SPIDER_ARCHIVE.is_file():
        urllib.request.urlretrieve(SPIDER_URL, SPIDER_ARCHIVE)

    with zipfile.ZipFile(SPIDER_ARCHIVE) as archive:
        for db_id in missing_db_ids:
            target = DB_ROOT / db_id / f"{db_id}.sqlite"
            target.parent.mkdir(parents=True, exist_ok=True)
            name = f"spider_data/database/{db_id}/{db_id}.sqlite"
            target.write_bytes(archive.read(name))

def check_time(deadline):
    if time.monotonic() > deadline:
        return 1
    return 0
def execute_sql(example, query):
    db_id = example["db_id"]
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    if not db_path.is_file():
        raise FileNotFoundError(db_path)

    database_uri = db_path.as_uri()
    database_uri += "?mode=ro"

    deadline = time.monotonic() + 5

    connection = sqlite3.connect(database_uri, uri=True)
    try:
        connection.set_progress_handler(check_time(deadline), 10000)
        cursor = connection.execute(query)
        return cursor.fetchall()
    finally:
        connection.close()


## Baseline (обычная генерация и SynCode)

Ниже используются `Dataset` и `post_process_answer` из кода авторов. Цикл прогона, сохранение результатов и проверки я написал для этого задания отдельно от статьи. Оба режима получают один и тот же промпт, модель и лимит токенов. Сначала запускаются 5 debug- примеров, затем 50 eval.

`text_match` сравнивает SQL как текст после условной нормализации. `executable` означает, что SQLite выполнил запрос без ошибки, но это не показатель правильности ответа. Ниже отдельно считается execution accuracy на заполненных базах Spider, скаченных с [huggingface](https://huggingface.co/datasets/HAL-9001/spider-databases).<br>
Результаты выполнения сравниваются напрямую, с учётом порядка только для эталона с `ORDER BY`.


In [18]:
import json
import re
from time import perf_counter

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_ID = "Qwen/Qwen3.5-0.8B"
GENERATORS = {"original": generate_original, "syncode": generate_syncode}


def baseline_messages(example):
    return [
        {"role": "system", "content": "Return exactly one SQLite SELECT query. No Markdown or explanation."},
        {"role": "user", "content": example["prompt"]},
    ]


def normalized_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().rstrip(";").strip()).casefold()


def run_baseline(examples, mode, split):
    path = RESULTS_DIR / f"{split}_{mode}.jsonl"
    by_id = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            row = json.loads(line)
            assert row["source_id"] not in by_id, f"Повтор ID в {path}"
            by_id[row["source_id"]] = row

    with path.open("a", encoding="utf-8") as output:
        for example in examples:
            source_id = example["source_id"]
            messages = baseline_messages(example)
            if source_id in by_id:
                row = by_id[source_id]
                assert (row["prompt"] == example["prompt"] and row.get("input_messages") == messages
                        and row["mode"] == mode
                        and row["model_id"] == MODEL_ID and row["max_new_tokens"] == MAX_NEW_TOKENS
                        and row.get("stop_words") == list(STOP_WORDS)),f"Настройки или выборка изменились: {path}, ID {source_id}"
                continue

            started = perf_counter()
            raw_output, tokens = GENERATORS[mode](messages)
            seconds = perf_counter() - started
            sql = sql_dataset.post_process_answer(raw_output).strip()
            try:
                execute_sql(example, sql)
                executable, sql_error = True, None
            except (sqlite3.Error, FileNotFoundError) as error:
                executable, sql_error = False, str(error)

            row = {
                "source_id": source_id,
                "db_id": example["db_id"],
                "question": example["question"],
                "prompt": example["prompt"],
                "input_messages": messages,
                "gold_sql": example["ground_truth"],
                "mode": mode,
                "model_id": MODEL_ID,
                "max_new_tokens": MAX_NEW_TOKENS,
                "stop_words": list(STOP_WORDS),
                "raw_output": raw_output,
                "sql": sql,
                "tokens": tokens,
                "seconds": seconds,
                "text_match": normalized_sql(sql) == normalized_sql(example["ground_truth"]),
                "executable": executable,
                "sql_error": sql_error,
            }
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
            output.flush()
            by_id[source_id] = row
            print(f"{split} {mode}: {len(by_id)}/{len(examples)}")

    return [by_id[example["source_id"]] for example in examples]


def print_summary(rows, mode):
    n = len(rows)
    print(f"{mode}: {n} примеров")
    print(f"  text_match: {sum(row['text_match'] for row in rows)}/{n}")
    print(f"  executable: {sum(row['executable'] for row in rows)}/{n}")
    print(f"  среднее число токенов: {sum(row['tokens'] for row in rows) / n:.1f}")
    print(f"  суммарное время: {sum(row['seconds'] for row in rows):.1f} с")


In [19]:
debug_results = {
    mode: run_baseline(debug_examples, mode, "debug") for mode in GENERATORS
}

for index, example in enumerate(debug_examples):
    print("\n" + "-" * 70)
    print("ID:", example["source_id"], "База:", example["db_id"])
    print("Вопрос:", example["question"])
    print("Эталонный SQL:", example["ground_truth"])
    for mode in GENERATORS:
        row = debug_results[mode][index]
        print(f"\n{mode} — полный ответ модели:\n{row['raw_output']}")
        print("SQL после обработки:", row["sql"])
        print("text_match:", row["text_match"], "executable:", row["executable"])
        print("Ошибка SQL:", row["sql_error"])
        print("Токенов:", row["tokens"], "Время, с:", round(row["seconds"], 2))



----------------------------------------------------------------------
ID: 1029 База: singer
Вопрос: What are the citizenships that are shared by singers with a birth year before 1945 and after 1955?
Эталонный SQL: select citizenship from singer where birth_year < 1945 intersect select citizenship from singer where birth_year > 1955

original — полный ответ модели:
SELECT DISTINCT s.citizenship
FROM db_singer s
JOIN db_song song ON s.singer_id = song.singer_id
WHERE s.birth_year < 1945 AND s.birth_year > 1955;

SQL после обработки: SELECT DISTINCT s.citizenship FROM db_singer s JOIN db_song song ON s.singer_id = song.singer_id WHERE s.birth_year < 1945 AND s.birth_year > 1955;
text_match: False executable: False
Ошибка SQL: no such table: db_singer
Токенов: 53 Время, с: 3.64

syncode — полный ответ модели:
SELECT DISTINCT s1citizenship
FROM singer AS s1
JOIN song AS s2 ON s1singer_id = s2singer_id
WHERE s1birth_year < 1945 AND s1birth_year > 1955;


SQL после обработки: SELECT DISTINC

In [20]:
eval_results = {
    mode: run_baseline(eval_examples, mode, "eval")
    for mode in GENERATORS
}

for mode, rows in eval_results.items():
    print_summary(rows, mode)

summary = {
    "model_id": MODEL_ID,
    "selection_seed": SEED,
    "eval_examples": len(eval_examples),
    "databases": selected_databases,
    "metrics": {
        mode: {
            "text_match": sum(row["text_match"] for row in rows),
            "executable": sum(row["executable"] for row in rows),
            "total": len(rows),
            "average_tokens": sum(row["tokens"] for row in rows) / len(rows),
            "total_seconds": sum(row["seconds"] for row in rows),
        }
        for mode, rows in eval_results.items()
    }
}
(RESULTS_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)


original: 50 примеров
  text_match: 0/50
  executable: 21/50
  среднее число токенов: 58.0
  суммарное время: 142.8 с
syncode: 50 примеров
  text_match: 0/50
  executable: 5/50
  среднее число токенов: 103.3
  суммарное время: 248.7 с


575

In [27]:
import sqlite3
import json
import re
from collections import Counter

def check_execution(example: dict, predicted_sql: str) -> dict:
    """Сравнивает результаты эталонного и предсказанного SQL-запросов."""
    gold_sql = example["ground_truth"]
    
    try:
        gold_rows = execute_sql(example, gold_sql)
    except sqlite3.Error as e:
        raise RuntimeError(f"Эталонный SQL не выполнился: {e}")

    try:
        predicted_rows = execute_sql(example, predicted_sql)
    except sqlite3.Error as error:
        return {
            "execution_match": False, 
            "executable": False, 
            "sql_error": str(error)
        }

    # Проверяем, есть ли сортировка в эталонном запросе
    has_order_by = bool(re.search(r"\border\s+by\b", gold_sql, re.IGNORECASE))
    
    if has_order_by:
        is_match = predicted_rows == gold_rows
    else:
        is_match = Counter(predicted_rows) == Counter(gold_rows)

    return {
        "execution_match": is_match,
        "executable": True,
        "sql_error": None,
        "gold_row_count": len(gold_rows),
        "predicted_row_count": len(predicted_rows),
    }

def evaluate_split(examples: list, results: dict, split_name: str) -> dict:
    """Оценивает результаты для конкретного прогона (eval/debug) по всем моделям"""
    scored_by_mode = {}
    
    for mode, rows in results.items():
        assert len(examples) == len(rows), "Количество примеров и результатов не совпадает"
        
        scored_rows = []
        for example, row in zip(examples, rows):
            assert example["source_id"] == row["source_id"], "Несовпадение ID источника"
            
            score = check_execution(example, row["sql"])
            scored_rows.append({
                "source_id": row["source_id"],
                "db_id": row["db_id"],
                "mode": mode,
                **score
            })
            
        # Сохраняем детальные результаты в JSONL
        file_path = RESULTS_DIR / f"{split_name}_execution_{mode}.jsonl"
        with file_path.open("w", encoding="utf-8") as f:
            for item in scored_rows:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                
        scored_by_mode[mode] = scored_rows
        
    return scored_by_mode



""" --- Основная логика выполнения --- """

eval_execution = evaluate_split(eval_examples, eval_results, "eval")
debug_execution = evaluate_split(debug_examples, debug_results, "debug")

for mode, rows in eval_execution.items():
    correct = sum(r["execution_match"] for r in rows)
    print(f"{mode} execution accuracy: {correct} / {len(rows)}")

for mode, rows in debug_execution.items():
    debug_info = [(r["source_id"], r["execution_match"]) for r in rows]
    print(f"{mode} debug execution: {debug_info}")

""" --- Обновление и сохранение summary --- """

summary["database_source"] = SPIDER_URL

for mode, rows in eval_execution.items():
    total = len(rows)
    correct = sum(r["execution_match"] for r in rows)
    executable = sum(r["executable"] for r in rows)
    
    summary["metrics"][mode]["execution_correct"] = correct
    summary["metrics"][mode]["execution_accuracy"] = correct / total if total > 0 else 0.0
    summary["metrics"][mode]["executable"] = executable

summary_path = RESULTS_DIR / "summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
    f.write("\n")

original execution accuracy: 5 / 50
syncode execution accuracy: 1 / 50
original debug execution: [(1029, False), (537, False), (399, False), (37, True), (295, True)]
syncode debug execution: [(1029, False), (537, False), (399, False), (37, False), (295, True)]


## Собственный контроллер IterGen: первые 5 примеров

Ниже — собственный цикл проверки и откатов. Библиотека авторов `IterGen` используется как механизм `start / forward / view / backward`; в локальной копии `itergen/itergen/main.py` добавлена совместимость с Transformers 5.3 и кэшем Qwen3.5. После отката Qwen пересчитывает состояние модели по сохранённому префиксу.

Проверяем завершённые имена таблиц и столбцов по схеме текущей базы. Сначала до двух раз откатываем только ошибочное имя. Если в том же месте ошибка повторяется, откатываемся к `select_expr` (выражению после `SELECT`). Общие ограничения шагов, откатов и токенов исключают бесконечные повторения.


In [ ]:
from itergen.main import IterGen

MAX_ITER_STEPS = 25
MAX_BACKTRACKS = 8
MAX_TOTAL_ITER_TOKENS = 300
LOCAL_RETRIES = 2


def schema_names(example):
    db_id = example["db_id"]
    db_path = DB_ROOT / db_id / f"{db_id}.sqlite"
    connection = sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)
    try:
        names = [name for (name,) in connection.execute(
            "SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"
        )]
        tables = {name.casefold() for name in names}
        columns = set()
        for name in names:
            quoted = name.replace('"', '""')
            columns.update(row[1].casefold() for row in connection.execute(
                f'PRAGMA table_info("{quoted}")'
            ))
        return tables, columns
    finally:
        connection.close()


def valid_name(unit, value, tables, columns):
    name = value.strip().strip('"`').casefold()
    if unit == "table_name":
        return name in tables
    # shortcut: check columns across the DB; resolve table and output aliases before stricter validation.
    column = name.rsplit(".", 1)[-1].strip('"`')
    return column == "*" or column in columns


def generate_itergen(example, generator):
    started = perf_counter()
    tables, columns = schema_names(example)
    generator.start(baseline_messages(example))
    trace = []
    local_attempts = {}
    backtracks = 0
    status = "step_limit"

    for step in range(1, MAX_ITER_STEPS + 1):
        if generator.finished():
            retained = generator.session_tokens.shape[-1] - generator.prompt_tokens.shape[-1]
            status = "max_output_tokens" if retained >= MAX_NEW_TOKENS else "stopped"
            break
        if generator._metadata["total_tokens"] >= MAX_TOTAL_ITER_TOKENS:
            status = "token_budget"
            break
        before = {unit: len(generator.view(unit)[0])
                  for unit in ("table_name", "column_name")}
        output = generator.forward(units=["table_name", "column_name"], num=1)[0]
        rejected = False
        for unit in ("table_name", "column_name"):
            new_names = generator.view(unit)[0][before[unit]:]
            for value in new_names:
                valid = valid_name(unit, value, tables, columns)
                event = {"step": step, "unit": unit, "value": value,
                         "valid": valid, "before": output}
                if valid:
                    event["action"] = "continue"
                else:
                    generator.backward(unit)
                    backtracks += 1
                    prefix = generator.structured_gen[0]
                    key = (unit, prefix)
                    local_attempts[key] = local_attempts.get(key, 0) + 1
                    event["action"] = f"backward_{unit}"
                    if local_attempts[key] > LOCAL_RETRIES and generator.view("select_expr")[0] \
                            and backtracks < MAX_BACKTRACKS:
                        generator.backward("select_expr")
                        backtracks += 1
                        event["action"] += "_then_select_expr"
                    event["after"] = generator.structured_gen[0]
                    rejected = True
                trace.append(event)
                if rejected:
                    break
            if rejected:
                break
        if backtracks >= MAX_BACKTRACKS:
            status = "backtrack_limit"
            break
        if not rejected and generator.finished():
            retained = generator.session_tokens.shape[-1] - generator.prompt_tokens.shape[-1]
            status = "max_output_tokens" if retained >= MAX_NEW_TOKENS else "stopped"
            break
    raw_output = generator.structured_gen[0]
    return {
        "raw_output": raw_output,
        "sql": sql_dataset.post_process_answer(raw_output).strip(),
        "trace": trace,
        "status": status,
        "backtracks": backtracks,
        "generated_tokens": generator._metadata["total_tokens"],
        "seconds": perf_counter() - started,
    }


In [ ]:
itergen_path = RESULTS_DIR / "debug_itergen.jsonl"
controller_config = {
    "controller_version": 2,
    "model_id": MODEL_ID, "max_new_tokens": MAX_NEW_TOKENS,
    "max_steps": MAX_ITER_STEPS, "max_backtracks": MAX_BACKTRACKS,
    "max_total_tokens": MAX_TOTAL_ITER_TOKENS, "local_retries": LOCAL_RETRIES,
    "recurrence_penalty": 0.3,
}
itergen_rows = {}
if itergen_path.exists():
    for line in itergen_path.read_text(encoding="utf-8").splitlines():
        row = json.loads(line)
        assert row["source_id"] not in itergen_rows
        itergen_rows[row["source_id"]] = row

pending = []
for example in debug_examples:
    row = itergen_rows.get(example["source_id"])
    if row is None:
        pending.append(example)
    else:
        assert row["prompt"] == example["prompt"] and row["controller_config"] == controller_config

if pending:
    iter_gen = IterGen(
        grammar="sql", model_id=MODEL_ID, device=device, quantize=False,
        do_sample=False, num_beams=1, num_return_sequences=1,
        max_new_tokens=MAX_NEW_TOKENS, max_tokens=1024,
        stop_strings=list(STOP_WORDS), recurrence_penalty=0.3,
    )
    if device.startswith("cuda"):
        iter_gen.model.to(dtype=torch.float16)
    iter_gen.model.eval()

    with itergen_path.open("a", encoding="utf-8") as output:
        for example in pending:
            source_id = example["source_id"]
            result = generate_itergen(example, iter_gen)
            row = {
                "source_id": source_id, "db_id": example["db_id"],
                "question": example["question"], "prompt": example["prompt"],
                "gold_sql": example["ground_truth"], "controller_config": controller_config,
                **result, **check_execution(example, result["sql"]),
            }
            output.write(json.dumps(row, ensure_ascii=False) + "\n")
            output.flush()
            itergen_rows[source_id] = row
            print(source_id, row["status"], "backtracks", row["backtracks"],
                  "tokens", row["generated_tokens"], "execution_match", row["execution_match"])

print("Отладочных примеров IterGen:", len(itergen_rows))


In [ ]:
ordered_itergen_rows = [itergen_rows[example["source_id"]] for example in debug_examples]
assert len(ordered_itergen_rows) == 5
assert all(0 < row["generated_tokens"] <= MAX_TOTAL_ITER_TOKENS + MAX_NEW_TOKENS
           and row["backtracks"] <= MAX_BACKTRACKS for row in ordered_itergen_rows)
assert all(check_execution(example, example["ground_truth"])["execution_match"]
           for example in debug_examples)

itergen_debug_summary = {
    "examples": len(ordered_itergen_rows),
    "execution_correct": sum(row["execution_match"] for row in ordered_itergen_rows),
    "execution_accuracy": sum(row["execution_match"] for row in ordered_itergen_rows) / len(ordered_itergen_rows),
    "total_backtracks": sum(row["backtracks"] for row in ordered_itergen_rows),
    "total_generated_tokens": sum(row["generated_tokens"] for row in ordered_itergen_rows),
    "statuses": {str(row["source_id"]): row["status"] for row in ordered_itergen_rows},
}
(RESULTS_DIR / "itergen_debug_summary.json").write_text(
    json.dumps(itergen_debug_summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("IterGen debug:", itergen_debug_summary)
for row in ordered_itergen_rows:
    print("\nID", row["source_id"], "SQL:", row["sql"])
    for event in row["trace"]:
        if not event["valid"]:
            print("  step", event["step"], event["unit"], repr(event["value"]),
                  "->", event["action"])
